# 10. Human review bound to the exact action

## Problem

A reviewer sees one title and Markdown payload. A later edit, changed source revision, expired proposal or replayed decision must not authorize different content.

## Learning objectives

Run a real LangGraph interrupt; inspect exact payloads; approve, edit or reject; bind canonical hashes and versions; validate edits; reject stale and repeated review.

Run top to bottom in a fresh kernel. The lab uses actual model calls, tools and temporary local storage. Configure the services before execution. Every plotted value declares its measurement kind.

In [ ]:
import os
from pathlib import Path
from tempfile import TemporaryDirectory
from IPython.display import display
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "pyproject.toml").exists())
os.chdir(ROOT)
scratch = TemporaryDirectory(prefix="northstar-course-")
os.environ["MPLCONFIGDIR"] = str(Path(scratch.name) / "matplotlib")
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"
import matplotlib
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from deep_agent_lab.settings import Settings
from deep_agent_lab.courseware import diagram, bars
settings = Settings( deep_lab_mode="live", langfuse_enabled=False,
                    deep_lab_state_dir=Path(scratch.name) / "state")
assert settings.deep_lab_mode == "live" and not settings.langfuse_enabled

## Concept explanation

## Existing Northstar review flow

Your repository already uses a Deep Agents interrupt on `save_brief`, with approve, edit, and reject decisions.

LangGraph persists state at interruption points so the application can resume the corresponding thread later.

The repository also verifies the current interrupt ID and rejects an already-resolved review request.

These are essential foundations.

This module adds stronger binding between the reviewer decision and the exact authorized payload.

## Exact-action approval

```
{
  "name": "save_brief",
  "arguments": {
    "title": "Recovery Readiness",
    "markdown": "Approved policy findings..."
  }
}
```

The approval should authorize this precise action, not a vague instruction such as "the agent may save a brief."

A cryptographic digest can bind the approval to a canonical representation of the payload and its scope.

This hash is a comparison mechanism. It is not a signature and does not authenticate the reviewer.

The complete approval process also needs authenticated reviewer identity, permission checks, immutable proposal storage, expiration, and an atomic state transition.

## Approval state machine

This design treats editing as creating a revised proposal requiring validation and, where policy requires, fresh approval.

That is stricter than letting any edit inherit the authorization of the original payload.

The server must check the proposed action against current permissions and evidence requirements even if a human has supplied the edited text.

## Transactional approval store

The application implements a focused SQLite review ledger. It uses the same database for review state and saved artifacts so the authorization transition and local write can be committed atomically.

The `action_id` is an idempotency key.

The application records the committed brief ID on the unique `review_actions.id` row in the same SQLite transaction.

### Transaction boundary

`BEGIN IMMEDIATE` obtains a SQLite write transaction before checking and updating the action.

The `SAVED` branch allows a repeated identical request to return the existing result.

A conflicting payload is rejected.

The local brief insert and review-state transition belong to one transaction.

The Northstar wrapper checks trusted caller scope before AgentService validates the current interrupt, payload and evidence revisions. The original loopback API remains a single-user lab. A live PostgreSQL revision check and the SQLite commit do not share a distributed transaction; strict live revision fencing needs an additional design.

The reviewer should not be able to call this function directly using an arbitrary `action_id`.

For a multi-user system, the database operation must also verify the tenant and authorized principal within the trusted service boundary.

## Stale approvals

Suppose the reviewer approves a brief citing BKP-7 revision 4.

Before saving, the corpus activates revision 5.

Your application must decide whether policy revision 4 remains valid for the approved brief.

For a report that claims to describe *current* policy, the safer default is to mark the proposal stale and request renewed review.

The revision check should occur immediately before the commit, using authoritative revision data.

In a distributed system, that final check and the relevant write should be protected against races using appropriate transaction isolation, conditional writes, or version fencing.

## Mental model

Read the following diagram as an application boundary: the model proposes actions while the runtime performs validated operations. The figure renders through Matplotlib in the notebook output.

In [ ]:
display(diagram('Human review bound to the exact action', ['Draft action', 'Pending immutable payload', 'Approve / edit / reject', 'Check digest + revisions', 'Atomic approved save', 'Saved artifact or conflict'], [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (2, 5)]))
plt.close('all')

## Implementation from scratch

Build the smallest inspectable mechanism before connecting it to the full application. The code below exposes state and checks the invariant that motivates this module.

In [ ]:
import hashlib,json,time
from dataclasses import dataclass,field

def canonical_payload(value):
    return json.dumps(value,sort_keys=True,separators=(",",":"),ensure_ascii=False)

def payload_digest(value):
    return hashlib.sha256(canonical_payload(value).encode()).hexdigest()

@dataclass
class LearnerProposal:
    payload: dict
    version: int = 1
    expires: float = field(default_factory=lambda:time.time()+300)
    status: str = "pending"
    approved_digest: str | None = None

def transition(proposal,decision,expected_version,expected_digest,now,edited=None):
    if proposal.status != "pending" or now >= proposal.expires:
        raise ValueError("Unavailable or expired proposal")
    if expected_version != proposal.version or expected_digest != payload_digest(proposal.payload):
        raise ValueError("Stale proposal")
    if decision == "edit":
        if edited is None:
            raise ValueError("Edited payload required")
        proposal.payload = dict(edited)
        proposal.version += 1
    elif decision not in {"approve","reject"}:
        raise ValueError("Unknown decision")
    proposal.status = "rejected" if decision == "reject" else "approved"
    proposal.approved_digest = payload_digest(proposal.payload) if decision != "reject" else None

def exact_replay(proposal,payload,version,now):
    if (proposal.status != "approved" or version != proposal.version or now >= proposal.expires
        or payload_digest(payload) != proposal.approved_digest):
        raise ValueError("Approval does not bind this exact action")
    return canonical_payload(payload)

prototype = LearnerProposal({"operation":"save_brief","markdown":"Inspected policy"})
transition(prototype,"approve",1,payload_digest(prototype.payload),now=time.time())
for content,version in [({**prototype.payload,"markdown":"changed"},1),(prototype.payload,2)]:
    try:
        exact_replay(prototype,content,version,time.time())
    except ValueError as error:
        print("Rejected replay",error)
assert exact_replay(prototype,prototype.payload,1,time.time())
prototype.expires = time.time()-1
try:
    exact_replay(prototype,prototype.payload,1,time.time())
except ValueError:
    print("Expired approval rejected")


In [ ]:
import hashlib,json
from deep_agent_lab.storage import BriefStore
store = BriefStore(Path(scratch.name)/"review-store")
revisions = {"BKP-7":"4"}
payload = {"operation":"save_brief","thread_id":"thread","title":"Readiness",
           "markdown":"Reviewed findings","revisions":revisions}
def digest(value):
    canonical = json.dumps(value,sort_keys=True,separators=(",",":"),ensure_ascii=False)
    return hashlib.sha256(canonical.encode()).hexdigest()
original = digest(payload)
changed = digest({**payload,"markdown":payload["markdown"]+"!"})
assert original != changed
proposal = store.propose("action-1","thread","Readiness","Reviewed findings",revisions)
assert proposal["digest"] == original
print("Review version:",proposal["version"],"Digest prefix:",original[:12])

## Framework implementation

The following cell executes the existing Deep Agents/LangGraph application or its actual integration boundary. Inspect the actual service configuration and returned observations in each cell.

In [ ]:
from deep_agent_lab.service import AgentService,RunRequest,ReviewRequest,Decision,open_checkpointer,ThreadConflict
with open_checkpointer(settings) as cp:
    service = AgentService(settings,cp)
    try:
        pending = service.run(RunRequest(prompt="Draft and save a restore brief.",thread_id="review-lab"))
        print("Observed save proposal:",pending.status,pending.steps,pending.answer)
        if not pending.reviews:
            pending = service.run(RunRequest(prompt="Submit the cited draft with save_brief. The runtime will collect human approval before executing the write.",thread_id="review-lab"))
        assert pending.reviews, "The live model did not submit a reviewable save action"
        review = pending.reviews[0]
        assert not pending.saved_brief_ids
        print("Exact proposed action:",review.action_requests[0])
        result = service.resume(pending.thread_id,ReviewRequest(interrupt_id=review.id,
            decisions=[Decision(type="approve")],payload_hashes=review.payload_hashes,
            proposal_versions=review.proposal_versions))
        saved = service.store.get(result.saved_brief_ids[0])
        assert saved.markdown == review.action_requests[0]["args"]["markdown"]
        try:
            service.resume(pending.thread_id,ReviewRequest(interrupt_id=review.id,decisions=[Decision(type="approve")]))
        except ThreadConflict:
            print("Expected: replayed interrupt rejected.")
        else:
            raise AssertionError("Repeated review accepted")
    finally:
        service.close()

## Code walkthrough

LangGraph resumes an interrupt with Command; it can reenter node code, so the side effect still needs application idempotency. AgentService records the pending action, source revisions, digest, version and expiry. It rechecks active revisions before authorizing. `save_reviewed` accepts only a matching approved payload and atomically inserts the brief and marks the action saved. An edit is an explicit reviewer approval of a newly validated payload and increments its version. Hashes compare content; they do not authenticate the reviewer. Identity and owner permissions are enforced by the capstone wrapper.

## Interactive experiments

Edit the independent proposal and observe the new hash and version. Under this lab's policy, the authorized reviewer's edit-and-submit approves the edited payload.

In [ ]:
store.decide("action-1","edit",revisions,expected_digest=original,expected_version=1,
             title="Edited readiness",markdown="Corrected findings")
updated = store.review("action-1")
assert updated["version"] == 2 and updated["digest"] != original
brief_id = store.save_reviewed("thread","Edited readiness","Corrected findings",revisions)
assert store.save_reviewed("thread","Edited readiness","Corrected findings",revisions) == brief_id
print("Stable artifact after duplicate save:",brief_id)

## Failure experiment

Deliberately violate a contract or remove a required condition. A caught, asserted failure is an expected result; an uncaught error means this notebook has not passed.

In [ ]:
store.propose("stale","thread-2","Title","Content",{"BKP-7":"4"})
for revisions_now,expected_hash in [({"BKP-7":"5"},None),({"BKP-7":"4"},"tampered")]:
    try:
        store.decide("stale","approve",revisions_now,expected_digest=expected_hash)
    except ValueError as error:
        print("Expected stale approval:",error)
    else:
        raise AssertionError("Stale evidence or payload accepted")

## Verification

These assertions inspect actual state produced above. They establish the stated contracts within this lab's scope.

In [ ]:
assert store.review("stale")["status"] == "PENDING"
assert store.review("action-1")["status"] == "SAVED"
with store.connect() as db:
    assert db.execute("SELECT count(*) FROM briefs").fetchone()[0] == 1
print("Verified exact payload, explicit edit binding, replay protection and idempotency.")

## Exercise

Reject a proposal and prove save_reviewed cannot commit it. Then expire a separate pending proposal and demonstrate that approval is denied.

Try your implementation before running the complete reference solution in the next cell.

### Reference solution

Compare the implementation and assertions with your attempt.

In [ ]:
store.propose("reject","thread-3","Title","Rejected",{})
store.decide("reject","reject",{})
try:
    store.save_reviewed("thread-3","Title","Rejected",{})
except PermissionError:
    pass
else:
    raise AssertionError("Rejected action saved")
store.propose("expired","thread-4","Title","Expired",{})
with store.connect() as db,db:
    db.execute("UPDATE review_actions SET expires=0 WHERE id='expired'")
try:
    store.decide("expired","approve",{})
except ValueError:
    print("Expired proposal denied.")
else:
    raise AssertionError("Expired action approved")

## Trade-offs and further reading

Approval authorizes an action; it does not prove every sentence is correct. The local SQL write and review state share a transaction. Live corpus activation in PostgreSQL is a separate database, so strict distributed revision fencing requires an additional transaction or immutable-snapshot policy; the lab does not claim cross-database atomicity.

[Primary source 1](https://docs.langchain.com/oss/python/langgraph/interrupts)

## Prediction and repair
Canonical JSON binds an action's content. A version binds the proposal revision and expiry
bounds its lifetime. The learner mechanism rejects changed content, newer versions and
expired replay before the real BriefStore demonstration.

## Framework mapping and transfer
BriefStore adds durable transactions, batch rollback, evidence revision checks and
idempotent saves to these transition checks. Inspect the persisted rows after each action.

## What this proves
The actual local store enforces exact-action approval and repeat-save identity.

## What this does not prove
The in-memory learner state supplies no cross-process transaction guarantee. Use the
repository store for integrated writes and durable recovery.


In [ ]:
plt.close('all')
scratch.cleanup()
print('All executed module assertions passed.')